# Imports

In [ ]:
import requests, re, json


## Barbora

### Testing scrape

#### Scraping a single prodcut

In [ ]:
# Test scraping single product

url = "https://barbora.lt/produktai/brandintas-jautienos-kumpio-didkepsnis-feel-the-beef-atsal-300-g"
headers = {"User-Agent": "Mozilla/5.0 (course-demo; personal use)"}

r = requests.get(url, headers=headers, timeout=20)
print(r.status_code, len(r.text))

m = re.search(r"window\.product\s*=\s*(\{.*?\});\s*</script>", r.text, re.S)
print("found:", bool(m))

if m:
    product = json.loads(m.group(1))
    print(product["title"])
    print(product["price"], product["comparative_unit_price"], product["comparative_unit"])

#### Scraping a page:

##### Single scape

In [ ]:
cat_url = "https://barbora.lt/mesa-zuvis-ir-kulinarija/sviezia-mesa-ir-paukstiena"

r = requests.get(cat_url, headers=headers, timeout=20)
print(r.status_code, len(r.text))

# 1. Is there embedded JSON with many products?
print("window.product(s) markers:", re.findall(r"window\.\w+\s*=", r.text)[:10])

# 2. How many product links are there?
links = re.findall(r'href="(/produktai/[^"]+)"', r.text)
print("product links found:", len(set(links)))
print(list(dict.fromkeys(links))[:5])

##### Find where products live

In [ ]:
for name in ["dataLayer", "currentPage", "b_data", "b_user_info"]:
    m = re.search(rf"window\.{name}\s*=\s*(.{{0,300}})", r.text, re.S)
    print(f"--- {name} ---")
    print(m.group(1) if m else None)
    print()

In [ ]:
html = r.text
print("'produktai/' occurrences:", html.count("produktai/"))
print("'sprandine' occurrences:", html.count("sprandine"))   # a word from a product you saw on this category page
print("'cnstrc' occurrences:", html.count("cnstrc"))

i = html.find("produktai/")
print(html[max(0, i-150): i+200] if i != -1 else "no match")

In [ ]:
def show_context(html, needle, width=200, max_hits=3):
    start = 0
    for n in range(max_hits):
        i = html.find(needle, start)
        if i == -1:
            break
        print(f"--- hit {n+1} for '{needle}' at position {i} ---")
        print(html[max(0, i-width): i+width].replace("\n", " "))
        print()
        start = i + 1

show_context(html, "sprandine", max_hits=2)
show_context(html, "cnstrc", max_hits=3)

In [ ]:
first = html.find('{"attributes":')
print("first product-like object at:", first)
print(html[max(0, first-300): first+100])

print("number of Url fields:", html.count('"Url":"'))

# the last window.xxx = ... assignment before the list
assigns = [(m.start(), m.group(0)) for m in re.finditer(r"window\.\w+\s*=", html) if m.start() < first]
print(assigns[-3:])

##### Products found, check the results

In [ ]:
marker = "window.b_productList ="
start = html.find(marker) + len(marker)

# skip whitespace to reach the '['
while html[start].isspace():
    start += 1

decoder = json.JSONDecoder()
products, end = decoder.raw_decode(html, start)

print(type(products), "with", len(products), "products")
print("keys of first product:")
print(sorted(products[0].keys()))

In [ ]:
for p in products[:5]:
    print(p["id"], "|", p["title"], "|", p["price"], "|",
          p["comparative_unit_price"], "/", p["comparative_unit"])